In [ ]:
import os
from utils_extraction import extract_body, tokenize, clean_tokens, decode
from utils_extraction import chunk_tokens, flatten_token_chunks
from utils_extraction import extract_few_shot_examples
from utils_extraction import select_few_shot 
from utils_extraction import merge_tokens_with_auto_labels, add_style_and_parent_to_auto_labels, compare_html_allow_auto_labels, correct_tokens_brackets, check_tokens_brackets
from models import GPTAssistant
from utils_extraction import process_chunks
from utils_extraction.html_utils import clean_html_formatting

In [2]:
# ---------- Define Hyperparameters ----------
min_tokens = 500
model_name = "gpt-4.1"

n_few_shot = 15  # Number of few-shot examples to use

#### Define the text to process, and where to save it. Define the text for few shot

In [3]:
# File paths
project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
filename = "2019SCC65"
round = "ronde_1"
anno = "llm"
version = "v1.0_2"
html_path = fr"{project_root}\data\Document_Échantillon_Initial\{round}\plain_html_arbre_balise\{filename}.htm"
output_dir = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}"

os.makedirs(output_dir, exist_ok=True)

# Read HTML file
with open(html_path, 'r', encoding='utf-8') as file:
    html_content = file.read()
print(f"   ✓ HTML file loaded: {html_path}")


fs_filename = "1999CanLII7320_annotated"
fs_anno = "EG"
fs_version = "v1"
fs_html_path = fr"{project_root}\data\Documents_Annotés\{fs_anno}\{fs_filename}_{fs_anno}_{fs_version}.html"
# Read HTML file
with open(fs_html_path, 'r', encoding='utf-8') as file:
    fs_html_content = file.read()
print(f"   ✓ HTML file loaded for few shot: {fs_html_path}")


   ✓ HTML file loaded: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Document_Échantillon_Initial\ronde_1\plain_html_arbre_balise\2019SCC65.htm
   ✓ HTML file loaded for few shot: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\EG\1999CanLII7320_annotated_EG_v1.html


### Process The HTML Content

In [4]:

# ---------- Extract body content ----------
body_content = extract_body(html_content)


# ---------- Tokenize body content ----------
tokens = tokenize(body_content)

# ---------- Clean tokens ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
token_chunks = chunk_tokens(normalized_cleaned_tokens, min_tokens=min_tokens, stop_bookmark_separation=True)


   ⚠ Warning: stop_bookmark_separation=True but bookmark not found
   ✓ Chunked tokens into 285 chunks (>= 500 tokens each)


In [5]:
# ---------- Extract body content ----------
fs_body_content = extract_body(fs_html_content)


# ---------- Tokenize body content ----------
fs_tokens = tokenize(fs_body_content)

# ---------- Clean tokens ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=fs_tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
token_chunk1, token_chunk2 = chunk_tokens(normalized_cleaned_tokens, min_tokens=min_tokens, stop_bookmark_separation=True)

   ✓ Found bookmark separator at index 23066
   ✓ Splitting: 23066 tokens before, 31001 tokens after
   ✓ Chunked tokens into 36 chunks (>= 500 tokens each)
   ✓ Chunked tokens into 62 chunks (>= 500 tokens each)
   ✓ Total chunks: 36 before + 62 after = 98


In [6]:
label_config = {
    "keep_attributes":["labelname"], # extraction only, no disambiguation
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    "keep_labels":["decision", "legislation", "secondary sources"]
}

In [7]:
# ---------- Create few-shot examples ----------

few_shot_examples = extract_few_shot_examples(token_chunk1, 
                                              label_config)



selected_few_shot_examples = select_few_shot(examples=few_shot_examples, n=n_few_shot)
print(f"   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")

   ✓ Extracted 36 few-shot examples from chunks
   ✓ Selected 15 few-shot examples for processing.


In [8]:
selected_few_shot_examples

[(" General Accident Assurance Company et al. v. Chrusz et al. Chrusz et al. v. General Accident Assurance Company et al. [Indexed as: General Accident Assurance Co. v. Chrusz] 45 O.R. (3d) 321 [1999] O.J. No. 3291 Docket No. C29463 Court of Appeal for Ontario Carthy, Doherty and Rosenberg JJ.A. September 14, 1999 Civil procedure -- Discovery -- Privilege -- Solicitor-client privilege -- Litigation privilege -- Common interest privilege -- Hotel destroyed by fire -- Insurance adjuster investigating fire -- Suspicion of arson -- Adjuster directed to provide reports directly to lawyer retained by insurer -- Insurer later making partial payments of insurance -- Subsequently, dismissed employee alleging that insured's claim fraudulent -- Insured's lawyer providing dismissed employee with copy of transcript of his statement -- Insurer suing insured -- Insured making counterclaim and joining employee -- Adjuster's reports before allegation of fraud not privileged -- Adjuster's reports after 

In [9]:
# ---------- Initialize LLM model ----------
model = GPTAssistant(model_name, temperature=1)

In [10]:
# ---------- Process chunks ----------

prompt_path = fr"{project_root}\llm_based_annotation\utils_extraction\prompts\simplified_parent_extraction_cot.txt"

processed_chunks = process_chunks(
    model=model,
    token_chunks=token_chunks,
    process_prompt_path=prompt_path,
    label_config=label_config,
    few_shot_examples=selected_few_shot_examples,
    output_dir=output_dir,
    filename=filename
)


   ✓ Processing 285 chunks with LLM...
   ✓ Using 15 few-shot examples


Processing chunks:  18%|█▊        | 51/285 [05:22<23:53,  6.13s/it]

   ⚠ Warning: <start> marker found but <end> marker missing


Processing chunks:  20%|██        | 57/285 [06:00<23:28,  6.18s/it]

   ⚠ Warning: <start> marker found but <end> marker missing


Processing chunks:  88%|████████▊ | 252/285 [26:10<03:20,  6.07s/it] 

   ⚠ Warning: <start> marker found but <end> marker missing


Processing chunks: 100%|██████████| 285/285 [29:15<00:00,  6.16s/it]

   ✓ Processing history saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2019SCC65\history_2019SCC65.json

   ✓ Processing completed:
      - Total chunks: 285
      - Successful: 285
      - Failed: 0
   ✓ Processed chunks saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2019SCC65\processed_chunks_2019SCC65.json


In [11]:
#write the processed chuncks in a json file for later use in the annotation interface

import json 
with open(f"{output_dir}\\processed_chunks_v2.json", "w") as f:
    json.dump(processed_chunks, f)

## Post Processing

In [4]:
# Read the processed_chuncks.json file to verify it was written correctly
import json
with open(f"{output_dir}\\processed_chunks.json", "r") as f:
    processed_chunks = json.load(f)

In [5]:
# Processed_chunks is a list of lists of tokens, we need to flatten it to get a single list of tokens for the whole document
processed_tokens_flat = flatten_token_chunks(processed_chunks)


# Read in parallel the original tokens and the processed tokens. Always prefer the original tokens, but if there is an auto_label token in the processed tokens, 
# we want to keep it and merge it with the original tokens. 
# This way we can keep the original formatting and structure of the document while adding the auto_labels extracted by the model.
original_tokens = tokenize(html_content)
processed_html_content_tokens = merge_tokens_with_auto_labels(original_tokens, processed_tokens_flat)

# This merging process can sometimes create some formatting issues with brackets, we need to correct them to get a valid HTML structure.
processed_html_content_tokens_corrected = correct_tokens_brackets(processed_html_content_tokens)
assert check_tokens_brackets(processed_html_content_tokens_corrected), "The brackets in the merged tokens are not balanced. Please check the merging and bracket correction steps for errors."



# The correction of the brackets can sometimes create some redoundant or useless formatting  with the HTML, we need to clean it to compare it with the original.
processed_html = decode(processed_html_content_tokens_corrected)
processed_html_cleaned = clean_html_formatting(processed_html)
print(f"\nMerged HTML length: {len(processed_html_cleaned)}")


# This step is just to ensure a good visualisation of HTMLLabelizer and to add the necessary attribute to stay consistent with the label scheme
processed_html_content = add_style_and_parent_to_auto_labels(processed_html_cleaned)


# Last check of the final processed_html_content with the original HTML, ignoring the auto_label tags which are not present in the original HTML but only in the processed one.
comparison_result = compare_html_allow_auto_labels(processed_html_content, html_content)
assert comparison_result, "The processed HTML content does not match the original HTML content when ignoring auto_label tags. Please check the merging and post-processing steps for errors."


# ---------- Save processed HTML to file ----------
with open(fr"{output_dir}\{filename}_llm_{version}.html", 'w', encoding='utf-8') as f:
    f.write(processed_html_content)
print(f"   ✓ Processed HTML saved to: {output_dir}")

   ✓ Flattened 285 chunks into 144800 tokens

Merged HTML length: 616429
   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)
   ✓ Processed HTML saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2019SCC65
